# 🧠 You-GPT v2: Google Colab Training Notebook
### Stage 2 Evolution: Response-Only Loss Masking & Dialogue-Turn Fine-Tuning

**What makes v2 superior to v1:**
1. **Response-Only Masking (`-100`)**: The model only trains on Parshwa's responses, not the friend's prompts.
2. **Pure WhatsApp Dialogue Turns**: Slices data by real conversation pairs, discarding the Takeout solo search queries.
3. **Controlled Epochs & Cosine LR**: Trains for ~5 epochs with warmup and decay to prevent rote memorization and learn *fluid style*.
4. **Fast T4 GPU Execution**: Runs in **~2 to 3 minutes** on free Colab GPU!

In [ ]:
# 1. Install dependencies
!pip install -q transformers tiktoken torch

In [ ]:
# 2. Check GPU
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. In Colab menu, go to Runtime > Change runtime type > T4 GPU!")

In [ ]:
# 3. Upload clean_chat_v2.txt (Combined WhatsApp + Instagram)
from google.colab import files
import os

target_file = 'clean_chat_v2.txt' if os.path.exists('clean_chat_v2.txt') else ('clean_chat.txt' if os.path.exists('clean_chat.txt') else None)
if not target_file:
    print("Please upload your 'clean_chat_v2.txt' file:")
    uploaded = files.upload()
    target_file = list(uploaded.keys())[0]
else:
    print(f"{target_file} already present!")

In [ ]:
# 4. Parse WhatsApp Dialogue Pairs & Mask Prompts
import re
import tiktoken

encoder = tiktoken.get_encoding("gpt2")

with open(target_file, 'r', encoding='utf-8') as f:
    text = f.read()

# Remove Takeout dump if present to keep pure WhatsApp peer dialogue
if '--- GOOGLE TAKEOUT' in text:
    text = text.split('--- GOOGLE TAKEOUT')[0]

raw_turns = []
for line in text.splitlines():
    m = re.match(r'^([^:\n]+):\s*(.*)', line)
    if m:
        spk, txt = m.group(1).strip(), m.group(2).strip()
        if spk and txt and not txt.startswith('---'):
            raw_turns.append((spk, txt))

# Form (Other Speaker -> Parshwa Nahar) dialogue pairs
pairs = []
for i in range(len(raw_turns) - 1):
    s1, t1 = raw_turns[i]
    s2, t2 = raw_turns[i+1]
    if s1 != 'Parshwa Nahar' and s2 == 'Parshwa Nahar':
        prompt = f"{s1}: {t1}\r\nParshwa Nahar:"
        response = f" {t2}\r\n"
        p_ids = encoder.encode(prompt)
        r_ids = encoder.encode(response)
        if len(p_ids) + len(r_ids) <= 256 and len(r_ids) > 0:
            pairs.append((p_ids, r_ids))

print(f"Extracted {len(pairs):,} clean WhatsApp dialogue pairs!")

# 90 / 10 Train-Val Split
split_idx = int(0.9 * len(pairs))
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]
print(f"Train pairs: {len(train_pairs):,} | Val pairs: {len(val_pairs):,}")

In [ ]:
# 5. Dataset & Collate Function with Response Masking
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

class ChatDataset(Dataset):
    def __init__(self, data_pairs):
        self.pairs = data_pairs
        
    def __len__(self):
        return len(self.pairs)
        
    def __getitem__(self, idx):
        p_ids, r_ids = self.pairs[idx]
        input_ids = p_ids + r_ids
        # Mask prompt tokens with -100 so loss is ONLY on the response!
        labels = [-100] * len(p_ids) + r_ids
        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

def collate_fn(batch):
    # Dynamic padding to max length in the batch
    inputs, targets = zip(*batch)
    max_len = max(len(x) for x in inputs)
    
    pad_inputs = []
    pad_targets = []
    for inp, tgt in zip(inputs, targets):
        pad_amt = max_len - len(inp)
        pad_inputs.append(F.pad(inp, (0, pad_amt), value=50256))  # 50256 = <|endoftext|>
        pad_targets.append(F.pad(tgt, (0, pad_amt), value=-100))  # -100 = ignore in loss
        
    return torch.stack(pad_inputs), torch.stack(pad_targets)

train_loader = DataLoader(ChatDataset(train_pairs), batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(ChatDataset(val_pairs), batch_size=16, shuffle=False, collate_fn=collate_fn)
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
# 6. Initialize Base GPT-2 Small & Optimizer
from transformers import GPT2LMHeadModel, GPT2Config, get_cosine_schedule_with_warmup
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}")

model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

# Hyperparameters
epochs = 5
learning_rate = 5e-5
total_steps = len(train_loader) * epochs

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

print(f"Total Epochs: {epochs} | Total Steps: {total_steps}")

In [ ]:
# 7. Training Loop with Validation
import time

@torch.no_grad()
def eval_val():
    model.eval()
    losses = []
    for xb, yb in val_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb, labels=yb)
        losses.append(out.loss.item())
    model.train()
    return sum(losses) / len(losses)

print("Starting You-GPT v2 Fine-Tuning...")
best_val_loss = float('inf')
start_time = time.time()

for ep in range(epochs):
    model.train()
    epoch_loss = 0
    for step, (xb, yb) in enumerate(train_loader):
        xb, yb = xb.to(device), yb.to(device)
        
        outputs = model(xb, labels=yb)
        loss = outputs.loss
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        epoch_loss += loss.item()
        
    train_loss = epoch_loss / len(train_loader)
    val_loss = eval_val()
    val_ppl = math.exp(val_loss)
    
    print(f"Epoch {ep+1:02d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'you_gpt_v2_weights.pth')
        print(f"  -> Saved new best checkpoint to 'you_gpt_v2_weights.pth'!")

print(f"\nTraining finished in {(time.time() - start_time) / 60:.1f} minutes!")

In [ ]:
# 8. Test Generation Sanity Check
model.eval()
test_prompt = "Darshan: bhai chal college jaana hai\r\nParshwa Nahar:"
inp = torch.tensor(encoder.encode(test_prompt), dtype=torch.long).unsqueeze(0).to(device)

with torch.no_grad():
    out = model.generate(
        inp,
        max_new_tokens=40,
        pad_token_id=50256,
        do_sample=True,
        temperature=0.65,
        top_p=0.85,
        repetition_penalty=1.08
    )

reply = encoder.decode(out[0].tolist()[inp.size(1):]).split('\n')[0].strip()
print(f"Prompt: {test_prompt.split('\r\n')[0]}")
print(f"Reply : {reply}")

In [ ]:
# 9. Download the trained v2 weights back to your laptop
files.download('you_gpt_v2_weights.pth')
print("Downloaded you_gpt_v2_weights.pth! Place it inside e:\you-gpt\model\ on your laptop.")